# 🏗️ Module 3: Model Architecture Deep Dive

In this notebook, we explore the transformer decoder architecture used in MiniMind.

**What you'll learn:**
- How the MiniMindConfig controls model size
- Parameter count breakdown by component
- Key architectural innovations: RMSNorm, SwiGLU, RoPE, and GQA
- How Mixture of Experts (MoE) works
- Scaling laws: how performance improves with model size

**Architecture overview**: MiniMind is a GPT-style decoder-only transformer with modern improvements from LLaMA/Mistral.

In [ ]:
# Install dependencies and set up environment
!pip install -q transformers torch matplotlib numpy
import subprocess, os

if not os.path.exists('/content/minimind-colab'):
    subprocess.run(['git', 'clone', 'https://github.com/Boyu-Zhang-UOI/minimind-colab', '/content/minimind-colab'])
    os.chdir('/content/minimind-colab')
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'])

import sys
sys.path.insert(0, '/content/minimind-colab')

!nvidia-smi

## ⚙️ MiniMind Configuration

The `MiniMindConfig` controls all architectural hyperparameters. Let's explore the default MiniMind-3 configuration.

In [ ]:
import sys
sys.path.insert(0, '/content/minimind-colab')
import torch
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

# Default MiniMind-3 configuration
config = MiniMindConfig(
    hidden_size=768,
    num_hidden_layers=8,
    num_attention_heads=8,
    num_key_value_heads=4,
    vocab_size=6400,
)

print("MiniMind-3 Configuration:")
print(f"  Hidden size (d_model):       {config.hidden_size}")
print(f"  Num layers:                  {config.num_hidden_layers}")
print(f"  Q heads:                     {config.num_attention_heads}")
print(f"  KV heads (GQA):              {config.num_key_value_heads}")
print(f"  Head dim:                    {config.hidden_size // config.num_attention_heads}")
print(f"  Vocab size:                  {config.vocab_size}")
print(f"  Intermediate size (FFN):     {config.intermediate_size}")
print(f"  Max position embeddings:     {config.max_position_embeddings}")
print(f"  RoPE base:                   {config.rope_theta}")

## 🔢 Parameter Count Analysis

A transformer has three main components per layer:
1. **Self-Attention**: Q, K, V projection matrices + output projection
2. **Feed-Forward Network (FFN)**: gate, up, and down projection matrices
3. **Layer Norms**: RMSNorm weights (tiny but important)

Plus: token embeddings and the final LM head.

In [ ]:
model = MiniMindForCausalLM(config)
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total/1e6:.2f}M\n")

print("Parameters by component:")
embed_params = model.model.embed_tokens.weight.numel()
print(f"  Token Embedding:  {embed_params/1e6:.2f}M ({embed_params/total*100:.1f}%)")

for i, layer in enumerate(model.model.layers[:2]):
    attn_params = sum(p.numel() for p in layer.self_attn.parameters())
    ffn_params = sum(p.numel() for p in layer.mlp.parameters())
    norm_params = (sum(p.numel() for p in layer.input_layernorm.parameters()) +
                   sum(p.numel() for p in layer.post_attention_layernorm.parameters()))
    print(f"  Layer {i}: Attention={attn_params/1e6:.3f}M | FFN={ffn_params/1e6:.3f}M | Norm={norm_params/1e3:.1f}K")

print(f"  ... (x{config.num_hidden_layers} layers total)")
lm_head_params = model.lm_head.weight.numel()
print(f"  LM Head: {lm_head_params/1e6:.2f}M (tied with embedding)")
print(f"\n  Per-layer params: ~{sum(p.numel() for p in model.model.layers[0].parameters())/1e6:.3f}M")

## 🔧 Key Architectural Innovations

### 1. RMSNorm (Root Mean Square Layer Normalization)

Standard LayerNorm: $\bar{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$

RMSNorm (simpler, faster): $\bar{x} = \frac{x}{\sqrt{\frac{1}{n}\sum x_i^2 + \epsilon}} \cdot \gamma$

RMSNorm removes the mean subtraction step, which is faster and performs similarly.

### 2. SwiGLU Activation (Feed-Forward Network)

Standard FFN: $\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$

SwiGLU FFN: $\text{FFN}(x) = (\text{Swish}(xW_{\text{gate}}) \odot xW_{\text{up}}) W_{\text{down}}$

where $\text{Swish}(x) = x \cdot \sigma(x)$

The gating mechanism lets the network learn which features to amplify or suppress.

### 3. RoPE (Rotary Position Embeddings)

Instead of adding absolute position embeddings to token embeddings, RoPE rotates the Q and K vectors by an angle proportional to position:

$q_m = q \cdot e^{im\theta}$, $k_n = k \cdot e^{in\theta}$

This makes attention score $q_m \cdot k_n$ depend only on *relative* position $(m - n)$, which generalizes better to longer sequences.

### 4. GQA (Grouped Query Attention)

Instead of one K,V head per Q head, multiple Q heads share a single K,V head:
- **MHA**: 8 Q + 8 K + 8 V heads
- **GQA (q=8, kv=4)**: 8 Q + 4 K + 4 V heads → saves 25% attention parameters

In [ ]:
# Inspect layer structure
layer = model.model.layers[0]
print("Layer 0 Architecture:")
print(f"  self_attn: {layer.self_attn.__class__.__name__}")
print(f"    q_proj: {list(layer.self_attn.q_proj.weight.shape)}")
print(f"    k_proj: {list(layer.self_attn.k_proj.weight.shape)}")
print(f"    v_proj: {list(layer.self_attn.v_proj.weight.shape)}")
print(f"    o_proj: {list(layer.self_attn.o_proj.weight.shape)}")
print(f"  mlp: {layer.mlp.__class__.__name__}")
print(f"    gate_proj: {list(layer.mlp.gate_proj.weight.shape)}")
print(f"    up_proj:   {list(layer.mlp.up_proj.weight.shape)}")
print(f"    down_proj: {list(layer.mlp.down_proj.weight.shape)}")
print(f"  input_layernorm:           {list(layer.input_layernorm.weight.shape)}")
print(f"  post_attention_layernorm:  {list(layer.post_attention_layernorm.weight.shape)}")

## 🔬 Grouped Query Attention (GQA)

In standard Multi-Head Attention (MHA), every query head has its own key and value head. GQA shares K,V heads across multiple Q heads.

```
MHA:  Q₁K₁V₁  Q₂K₂V₂  Q₃K₃V₃  Q₄K₄V₄  (4 independent heads)
GQA:  Q₁      Q₂      →  K₁V₁   (2 Q heads share K,V)
      Q₃      Q₄      →  K₂V₂
```

**Benefits:**
- Reduces KV cache size during inference (important for long sequences)
- Reduces attention parameter count
- Similar quality to MHA with proper training

In [ ]:
# Demonstrate GQA parameter savings
q_heads = config.num_attention_heads     # 8
kv_heads = config.num_key_value_heads    # 4
hidden = config.hidden_size              # 768
head_dim = hidden // q_heads             # 96

# MHA: Q(8 heads) + K(8 heads) + V(8 heads) + O(8 heads)
mha_params = hidden * (q_heads * head_dim + q_heads * head_dim + q_heads * head_dim + q_heads * head_dim)

# GQA: Q(8 heads) + K(4 heads) + V(4 heads) + O(8 heads)
gqa_params = hidden * (q_heads * head_dim + kv_heads * head_dim + kv_heads * head_dim + q_heads * head_dim)

print(f"Attention head configuration:")
print(f"  Q heads: {q_heads}, KV heads: {kv_heads}, head_dim: {head_dim}")
print(f"\nFull MHA parameters:             {mha_params/1e6:.3f}M")
print(f"GQA parameters (kv={kv_heads}):        {gqa_params/1e6:.3f}M")
print(f"Parameter reduction:             {(mha_params - gqa_params)/mha_params*100:.1f}%")

# Verify against actual model
actual_attn_params = sum(p.numel() for p in model.model.layers[0].self_attn.parameters())
print(f"\nActual model attention params:   {actual_attn_params/1e6:.3f}M (matches GQA formula ✓)")

## 🌀 RoPE (Rotary Position Embeddings)

RoPE encodes position information by rotating query and key vectors. The rotation frequency decreases with dimension index, creating a "clock" pattern across dimensions.

For position $m$ and dimension $i$:
$$\theta_i = \frac{1}{\text{base}^{2i/d}}$$

$$q_m^{(2i, 2i+1)} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} q^{(2i, 2i+1)}$$

Early dimensions rotate fast (short-range patterns), later dimensions rotate slow (long-range patterns).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dim = 64
end = 100
rope_base = 1e6  # MiniMind uses 1M base for longer context

freqs = 1.0 / (rope_base ** (np.arange(0, dim, 2)[:dim//2].astype(float) / dim))
positions = np.arange(end)
cos_vals = np.cos(np.outer(positions, freqs))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot cosine components for several dimensions
ax = axes[0]
for i in range(0, min(8, dim//2), 2):
    ax.plot(positions, cos_vals[:, i], label=f'dim {i}', alpha=0.8)
ax.set_title('RoPE Cosine Components\n(first few dimensions)')
ax.set_xlabel('Token Position')
ax.set_ylabel('cos(position × frequency)')
ax.legend(ncol=2)
ax.grid(True, alpha=0.3)

# Heatmap of all dimensions
ax = axes[1]
im = ax.imshow(cos_vals.T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_title('RoPE Cosine Heatmap\n(all dimensions)')
ax.set_xlabel('Token Position')
ax.set_ylabel('Dimension Index')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()
print(f"RoPE base: {rope_base:.0e} (higher base → better long-context generalization)")

## ⚡ SwiGLU Activation Function

SwiGLU replaces the standard ReLU activation in the FFN with a gated activation:

$$\text{SwiGLU}(x, W, V) = \text{Swish}(xW) \odot xV$$
$$\text{Swish}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

The gate ($xW$ through Swish) acts as a soft filter, deciding which features from the up-projection ($xV$) to pass through.

**Why it works better than ReLU:**
- Smooth gradients everywhere (unlike ReLU's dead neuron problem)
- Gating allows multiplicative interactions between features
- Empirically ~1-2% better perplexity at no extra inference cost

In [ ]:
layer = model.model.layers[0]
print("Feed-Forward Network (SwiGLU):")
print(f"  Input:     [batch, seq, {config.hidden_size}]")
print(f"  gate_proj: [{config.hidden_size}] → [{config.intermediate_size}]   (Swish activation)")
print(f"  up_proj:   [{config.hidden_size}] → [{config.intermediate_size}]   (linear)")
print(f"  ↓ element-wise multiply: gate ⊙ up")
print(f"  down_proj: [{config.intermediate_size}] → [{config.hidden_size}]  (project back)")
print(f"\n  Expansion ratio: {config.intermediate_size/config.hidden_size:.2f}x")
print(f"  FFN parameters: {sum(p.numel() for p in layer.mlp.parameters())/1e6:.3f}M per layer")

# Visualize Swish vs ReLU
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-4, 4, 200)
swish = x / (1 + np.exp(-x))
relu = np.maximum(0, x)
gelu = x * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

plt.figure(figsize=(8, 4))
plt.plot(x, swish, label='Swish (used in SwiGLU)', linewidth=2)
plt.plot(x, relu, label='ReLU', linewidth=2, linestyle='--')
plt.plot(x, gelu, label='GELU', linewidth=2, linestyle=':')
plt.grid(True, alpha=0.3)
plt.legend()
plt.title('Activation Functions Comparison')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.tight_layout()
plt.show()

## 🔀 Mixture of Experts (MoE) Variant

MiniMind supports an optional **Mixture of Experts (MoE)** architecture where the FFN is replaced by multiple "expert" FFNs, with a learned router deciding which experts to use for each token.

```
Standard FFN:  token → FFN → output
MoE FFN:       token → Router → [Expert 1 (weight 0.7), Expert 3 (weight 0.3)] → weighted sum → output
```

**Benefits:**
- Total parameters increase (more experts = more capacity)
- Active parameters per token stay the same (only top-k experts activate)
- The model can specialize different experts for different types of knowledge

In [ ]:
from trainer.trainer_utils import get_model_params

print("=== Dense MiniMind-3 ===")
dense_config = MiniMindConfig(
    hidden_size=768, num_hidden_layers=8, use_moe=False
)
dense_model = MiniMindForCausalLM(dense_config)
dense_params = sum(p.numel() for p in dense_model.parameters())
print(f"Total parameters: {dense_params/1e6:.2f}M")

print("\n=== MoE MiniMind-3 (4 experts, top-1) ===")
moe_config = MiniMindConfig(
    hidden_size=768, num_hidden_layers=8,
    use_moe=True, num_experts=4, num_experts_per_tok=1
)
moe_model = MiniMindForCausalLM(moe_config)
moe_params = sum(p.numel() for p in moe_model.parameters())
print(f"Total parameters: {moe_params/1e6:.2f}M")

print(f"\nMoE has {moe_params/dense_params:.2f}x more total parameters")
print(f"But only activates ~{1/moe_config.num_experts:.0%} of FFN params per token")
del dense_model, moe_model

## 📏 Experiment: Scaling Laws

Neural language models follow approximate scaling laws: performance improves predictably with model size, data, and compute.

Let's explore how parameter count scales with architecture choices.

In [ ]:
import gc

configs_to_test = [
    (512, 8, "MiniMind-Tiny"),
    (768, 8, "MiniMind-3 (default)"),
    (768, 12, "MiniMind-3 (deeper)"),
    (1024, 12, "MiniMind-Large"),
]

print(f"{'Config':<25} {'Hidden':>8} {'Layers':>7} {'Params':>10} {'Relative':>10}")
print("-" * 65)

base_params = None
for hidden, layers, name in configs_to_test:
    cfg = MiniMindConfig(hidden_size=hidden, num_hidden_layers=layers)
    m = MiniMindForCausalLM(cfg)
    params = sum(p.numel() for p in m.parameters())
    if base_params is None:
        base_params = params
    print(f"{name:<25} {hidden:>8} {layers:>7} {params/1e6:>8.1f}M {params/base_params:>9.2f}x")
    del m
    gc.collect()

## 📝 Student Exercise

**Calculate parameter count by hand:**

For a single transformer layer with:
- hidden_size = 768, head_dim = 96
- Q heads = 8, KV heads = 4
- intermediate_size = 2048

1. Attention parameters:
   - Q_proj: 768 × (8 × 96) = ?
   - K_proj: 768 × (4 × 96) = ?
   - V_proj: 768 × (4 × 96) = ?
   - O_proj: (8 × 96) × 768 = ?

2. FFN parameters:
   - gate_proj: 768 × 2048 = ?
   - up_proj:   768 × 2048 = ?
   - down_proj: 2048 × 768 = ?

3. Total per layer = attention + FFN + norms (tiny)

Check your answer against the actual model above!

## 💡 Discussion Questions

1. **Dense vs MoE tradeoffs:**
   - When would you prefer MoE over dense? (Hint: think about inference speed vs capacity)
   - What are the challenges of training MoE models? (Hint: load balancing)

2. **Why does GQA work?**
   - Queries need to be diverse (different things to attend to)
   - Keys/Values are more similar across heads (they represent the same tokens)
   - The KV cache grows with sequence length - why does this matter for deployment?

3. **RoPE base frequency:**
   - MiniMind uses `rope_theta=1e6`. What happens with a smaller base like 1e4?
   - Why do models like LLaMA extend to longer contexts by increasing this value?

4. **SwiGLU has 3 weight matrices per FFN layer but standard FFN has 2. Is this actually "free"?**
   - SwiGLU typically uses a smaller intermediate_size to keep parameter count similar
   - The intermediate size is often set to `2/3 × 4 × hidden_size` instead of `4 × hidden_size`